In [1]:
from classes.multilabel_classifier import MultiLabelClassifier
from testclassifier.model import LSTM_MultiLabel
import torch.nn as nn
# Wrapper to adapt LSTM_MultiLabel for non-rolling use
class LSTM_MultiLabel_SingleStep(nn.Module):
    """
    Wrapper around LSTM_MultiLabel to handle single timestep inputs.

    MultiLabelClassifier feeds single samples [batch, features], but LSTM expects
    [batch, seq_len, features]. This wrapper adds a sequence dimension.
    """
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.lstm_model = LSTM_MultiLabel(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            output_dim=output_dim,
            num_layers=num_layers,
            dropout=dropout,
            bidirectional=bidirectional
        )

    def forward(self, x):
        # x is [batch, features] from MultiLabelClassifier
        # Add sequence dimension: [batch, 1, features]
        x = x.unsqueeze(1)
        return self.lstm_model(x)
target_names = ["TWF", "HDF", "PWF", "OSF", "RNF"]

model = MultiLabelClassifier(
        module=LSTM_MultiLabel_SingleStep,
        label_names=target_names,
        optimizer_fn="adam",
        lr=1e-3,
        device='cuda:1',
        hidden_dim=128,
        num_layers=2,
        dropout=0.3,
        bidirectional=True,
        seed=42,
    )


model

MultiLabelClassifier (
  module=LSTM_MultiLabel_SingleStep(
  (lstm_model): LSTM_MultiLabel(
    (lstm): LSTM(1, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
    (dropout): Dropout(p=0.3, inplace=False)
    (fc): Linear(in_features=256, out_features=5, bias=True)
  )
)
  label_names=['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
  optimizer_fn="adam"
  lr=0.001
  device="cuda:1"
  seed=42
  thresholds={'TWF': 0.5, 'HDF': 0.5, 'PWF': 0.5, 'OSF': 0.5, 'RNF': 0.5}
  pos_weight=None
)

## Loading the predictive maintenance problem as a data stream

This repository allows the use of three public multi-label problems, providing the interface for loading them as streams. Specifically, the three problems are the following:

* The **AI4I 2020 Predictive Maintenance Dataset**: A synthetic dataset that reflects real predictive maintenance data encountered in industry. [[Reference]](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset).
* The **Naval Propulsion System (NPS) Dataset**: Data generated from a sophisticated simulator of a Gas Turbines (GT), mounted on a Frigate characterized by a COmbined Diesel eLectric And Gas (CODLAG) propulsion plant type. [[Reference]](https://archive.ics.uci.edu/dataset/316/condition+based+maintenance+of+naval+propulsion+plants).
* The **Alarms Logs in Packaging Industry (ALPI)** dataset consists of a sequence of alarms logged by packaging equipment in an industrial environment. The collection includes data logged by 20 machines, deployed in different plants around the world, from 2019-02-21 to 2020-06-17. There are 154 distinct alarm codes, whose distribution is highly unbalanced. In our implementation, the stream is loaded by machine, and the length of the rolling windows for input and output can be also specified. The machine ID is an integer in [0, 19]. [[Reference]](https://ieee-dataport.org/open-access/alarm-logs-packaging-industry-alpi).

In [2]:
from datasets.multioutput import *

stream = Ai4i()
stream

The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.

Source: https://archive.ics.uci.edu/ml/datasets/AI4I+2020+Predictive+Maintenance+Dataset

      Name  Ai4i                                                                                      
      Task  Multi-output binary classification                                                        
   Samples  10,000                                                                                    
  Features  14                                                                                        
   Outputs  5                                                                                         
    Sparse  False                                                                                     
      Path  /home/quique/river_data/Ai4i/ai4i2020.csv                                                 
       URL  https://archive.ics.uci.edu/static/p

### Access to the stream

Using Python iterators one can access to the stream sample-by-sample. Both input and multi-label output are provided in a dictonary-based format.

In [3]:
# Using Ai4i as data stream

x, y = next(iter(stream))
x

{'Type': 'M',
 'Air temperature [K]': 298.1,
 'Process temperature [K]': 308.6,
 'Rotational speed [rpm]': 1551.0,
 'Torque [Nm]': 42.8,
 'Tool wear [min]': 0.0}

In [4]:
y

{'TWF': False, 'HDF': False, 'PWF': False, 'OSF': False, 'RNF': False}

In [6]:
import evaluate
import numbers
import torch
import sys
from river.compose import SelectType
from river.metrics import F1
from river.metrics.base import Metrics
from river.metrics.multioutput import ExactMatch
from river import preprocessing
from custommetrics.multioutput import *

## Running the model without rolling



In [7]:



pp = SelectType(numbers.Number) | preprocessing.StandardScaler()
pp += SelectType(str) |  preprocessing.OneHotEncoder()
pipeline = pp | model

evaluate.progressive_val_score(dataset=stream, model=pipeline, metric=Metrics([ExactMatch(), MacroAverage(F1()), MicroAverage(F1())]), show_memory=True, print_every=100)

[100] ExactMatch: 84.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 97.91 KiB
[200] ExactMatch: 90.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 97.91 KiB
[300] ExactMatch: 91.67%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[400] ExactMatch: 93.25%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[500] ExactMatch: 94.20%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[600] ExactMatch: 95.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[700] ExactMatch: 95.57%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[800] ExactMatch: 96.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[900] ExactMatch: 96.22%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[1,000] ExactMatch: 96.40%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[1,100] ExactMatch: 96.36%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 98.02 KiB
[1,200] ExactMatch: 96.17%
MacroAverage(F1): 0.00%
MicroAvera

ExactMatch: 96.55%
MacroAverage(F1): 19.05%
MicroAverage(F1): 29.01%

## Running the model with rolling

In [6]:


import torch

from classes.rolling_multilabel_classifier import RollingMultiLabelClassifier
import numbers
device = "cuda:0" if torch.cuda.is_available() else "cpu"
window_size = 100


clf = RollingMultiLabelClassifier(
        module=LSTM_MultiLabel,
        label_names=target_names,
        optimizer_fn="adam",
        lr=1e-3,
        device=device,
        window_size=window_size,
        append_predict=False,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3,
        bidirectional=True,
        output_dim=len(target_names),
        seed=42,
        epochs=10
    )
pr = SelectType(numbers.Number) | preprocessing.StandardScaler()
pr += SelectType(str) |  preprocessing.OneHotEncoder()
pipeliner = pr | clf

evaluate.progressive_val_score(dataset=stream, model=pipeliner, metric=Metrics([ExactMatch(), MacroAverage(F1()), MicroAverage(F1())]), show_memory=True, print_every=100)

⚠ Warning: New features detected in predict (6 -> 7). Reinitializing model...
⚠ Warning: New features detected in predict (7 -> 8). Reinitializing model...
[100] ExactMatch: 97.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 117.86 KiB
[200] ExactMatch: 95.50%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.75 KiB
[300] ExactMatch: 94.67%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.86 KiB
[400] ExactMatch: 95.25%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.34 KiB
[500] ExactMatch: 95.80%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.86 KiB
[600] ExactMatch: 96.33%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.34 KiB
[700] ExactMatch: 96.71%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.86 KiB
[800] ExactMatch: 96.87%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.34 KiB
[900] ExactMatch: 97.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 120.86 KiB
[1,000] ExactMatch: 96.90%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0

ExactMatch: 95.79%
MacroAverage(F1): 16.14%
MicroAverage(F1): 27.00%

In [4]:

from classes.pretrained_rolling_classifier import PretrainedRollingMultiLabelClassifier

device = "cuda:0" if torch.cuda.is_available() else "cpu"
window_size = 1000

    # Rutas al modelo preentrenado
checkpoint_path = "testbatch/lstm_multilabel_ai4i_complete.pt"
scaler_path =  "testbatch/scaler_ai4i.pkl"

    # === Pretrained Rolling multi-label classifier ===
clft = PretrainedRollingMultiLabelClassifier(
        module=LSTM_MultiLabel,
        label_names=target_names,
        checkpoint_path=str(checkpoint_path),
        scaler_path=str(scaler_path),
        optimizer_fn="adam",
        lr=1e-4,  # Learning rate más bajo para fine-tuning
        device=device,
        window_size=window_size,  # Según el modelo preentrenado
        append_predict=False,
        freeze_pretrained=False,  # Permitir fine-tuning
        hidden_dim=64,
        num_layers=2,
        dropout=0.2,
        bidirectional=True,
        epochs=10,
    )

pipelinet = ""
pipelinet =  clft

evaluate.progressive_val_score(dataset=stream, model=pipelinet, metric=Metrics([ExactMatch(), MacroAverage(F1()), MicroAverage(F1())]), show_memory=True, print_every=100)

/home/quique/tesis/deep-river-tesis/.venv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✓ Loaded pretrained model from: testbatch/lstm_multilabel_ai4i_complete.pt
  - Total parameters: 137,861
  - Trainable parameters: 137,861
  - Frozen parameters: 0
  - Pretrained epochs: 100
  - Pretrained final loss: 0.0004
[100] ExactMatch: 97.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 115 KiB
[200] ExactMatch: 96.50%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 145.99 KiB
[300] ExactMatch: 96.00%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 177.49 KiB
[400] ExactMatch: 96.50%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 208.47 KiB
[500] ExactMatch: 96.80%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 239.97 KiB
[600] ExactMatch: 97.17%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 270.95 KiB
[700] ExactMatch: 97.43%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 302.45 KiB
[800] ExactMatch: 97.63%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 333.44 KiB
[900] ExactMatch: 97.67%
MacroAverage(F1): 0.00%
MicroAverage(F1): 0.00% – 364.94 KiB
[1,0

ExactMatch: 96.57%
MacroAverage(F1): 20.94%
MicroAverage(F1): 30.71%